In [0]:
%sql
CREATE CATALOG IF NOT EXISTS SCD_PRAC

In [0]:
%sql
create schema if not exists SCD_PRAC.bronze;
create schema if not exists SCD_PRAC.silver;

In [0]:
%sql
create volume if not exists SCD_PRAC.bronze.data

In [0]:
%sql

create table if not exists SCD_PRAC.bronze.data(
  customer_id int,
  customer_name string,
  city string,
  status string
)


In [0]:
from pyspark.sql.functions import *

df = spark.read.format("csv").option("header",True).load("/Volumes/scd_prac/bronze/data/customers_day1.csv")
df.write.mode("append").saveAsTable("SCD_PRAC.bronze.data")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS SCD_PRAC.silver.dim_customer (
 customer_sk INT,
 customer_id INT,
 customer_name STRING,
 city STRING,
 status STRING,
 effective_start_date DATE,
 effective_end_date DATE,
 is_current BOOLEAN)

In [0]:
sk = spark.tabel(SCD_PRAC.silver.dim_customer).agg(max(customer_sk)).collect()[][]
sk_key = 1 when sk is null else sk+1

In [0]:

MERGE into SCD_PRAC.silver.dim_customer a
using SCD_PRAC.sbronze.dim_data b
on a.id = b.id and a.is_current = "True"
when matched and (a.customer_name <> b.customer_name or a.city <> b.city or a.status <> b.status)
then update set a.effective_end_date = current_date(), a.is_current = "False"
when not matched then insert(
 customer_sk,
 customer_id,
 customer_name,
 city,
 status,
 effective_start_date,
 effective_end_date,
 is_current
)
values(
(select when (customer_sk is null,1,max(customer_sk)+1 from SCD_PRAC.silver.dim_customer),
b.customer_id,
b.customer_name.
b.city,
b.status,
current_date(),
"9999-12-31",
"True"
)